# 03 Graphics

Динамический график с event detector и целевыми уровнями вокруг события.

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from src.connector.data_fetcher import load_all_price_data
from src.features.feature_pipeline import generate_features
from src.features.event_detector import detect_events

## Параметры

Все параметры локальные для этого notebook.

In [4]:
# Если даты None, берем последние 2 месяца данных.
date_start = None  # пример: "2026-01-01"
date_finish = None  # пример: "2026-03-01"

# Вертикальная дистанция от event close до верхней/нижней полоски.
THRESHOLD = 0.0005

# Горизонтальная длина полосок в свечах.
HORIZON = 8

In [5]:
# Загружаем и готовим данные.
price_df, loaded_files = load_all_price_data(PROJECT_ROOT / "data")
df = detect_events(generate_features(price_df)).sort_index()

if date_finish is None:
    finish = df.index.max()
else:
    finish = pd.Timestamp(date_finish)

if date_start is None:
    start = finish - pd.DateOffset(months=2)
else:
    start = pd.Timestamp(date_start)

if df.index.tz is not None:
    if start.tzinfo is None:
        start = start.tz_localize(df.index.tz)
    if finish.tzinfo is None:
        finish = finish.tz_localize(df.index.tz)

view_df = df.loc[(df.index >= start) & (df.index <= finish)].copy()
events = view_df[view_df["event"].eq(1)].copy()

print(f"CSV файлов: {len(loaded_files)}")
print(f"Период графика: {view_df.index.min()} -> {view_df.index.max()}")
print(f"Свечей на графике: {len(view_df):,}")
print(f"Events на графике: {len(events):,}")

CSV файлов: 45
Период графика: 2026-01-26 16:30:00+00:00 -> 2026-03-26 16:30:00+00:00
Свечей на графике: 4,133
Events на графике: 341


## График

Точки — найденные events. Зеленая полоска сверху и красная снизу показывают расстояние `THRESHOLD` на следующие `HORIZON` свечей.

In [6]:
def candle_at_offset(index: pd.Index, timestamp: pd.Timestamp, offset: int) -> pd.Timestamp:
    pos = index.get_indexer([timestamp], method="nearest")[0]
    target_pos = min(pos + offset, len(index) - 1)
    return index[target_pos]


def add_event_levels(fig: go.Figure, source_df: pd.DataFrame, event_df: pd.DataFrame) -> None:
    for event_time, row in event_df.iterrows():
        close = float(row["close"])
        x0 = event_time
        x1 = candle_at_offset(source_df.index, event_time, HORIZON)
        upper = close * (1 + THRESHOLD)
        lower = close * (1 - THRESHOLD)

        fig.add_shape(
            type="line",
            x0=x0,
            x1=x1,
            y0=upper,
            y1=upper,
            line=dict(color="#16a34a", width=2),
            opacity=0.85,
        )
        fig.add_shape(
            type="line",
            x0=x0,
            x1=x1,
            y0=lower,
            y1=lower,
            line=dict(color="#dc2626", width=2),
            opacity=0.85,
        )


fig = go.Figure()

fig.add_trace(
    go.Candlestick(
        x=view_df.index,
        open=view_df["open"],
        high=view_df["high"],
        low=view_df["low"],
        close=view_df["close"],
        name="EURUSD",
        increasing_line_color="#0f9f6e",
        decreasing_line_color="#e11d48",
    )
)

fig.add_trace(
    go.Scatter(
        x=events.index,
        y=events["close"],
        mode="markers",
        name="Event detector",
        marker=dict(
            size=13,
            color="#facc15",
            line=dict(color="#111827", width=2),
            symbol="circle",
        ),
        customdata=np.stack(
            [events["event_cusum_direction"].fillna(0).to_numpy(), events["close"].to_numpy()],
            axis=-1,
        ) if not events.empty else None,
        hovertemplate="Event<br>%{x}<br>close=%{customdata[1]:.5f}<br>direction=%{customdata[0]}<extra></extra>",
    )
)

add_event_levels(fig, view_df, events)

fig.update_layout(
    title=f"EURUSD events | THRESHOLD={THRESHOLD:.5f}, HORIZON={HORIZON} candles",
    height=780,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(l=20, r=20, t=80, b=30),
    xaxis=dict(
        rangeslider=dict(visible=True),
        rangeselector=dict(
            buttons=list([
                dict(count=7, label="7d", step="day", stepmode="backward"),
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=2, label="2m", step="month", stepmode="backward"),
                dict(step="all", label="all"),
            ])
        ),
    ),
    yaxis=dict(title="Price", fixedrange=False),
)

fig.show()